# Data Cleaning & Preprocessing

## Objective

Data cleaning is a critical step in the data analysis process. The quality of insights depends heavily on the quality of the underlying data.

This notebook focuses on identifying and resolving data quality issues, improving feature readability, and preparing the dataset for exploratory analysis.

---

## Cleaning Tasks

- Load the raw dataset
- Rename columns for readability
- Inspect missing values
- Handle missing values
- Check duplicate records
- Validate data types
- Perform data quality checks
- Export the cleaned dataset

## Table of Contents

1. Import Libraries
2. Load Dataset
3. Rename Columns
4. Missing Value Analysis
5. Handle Missing Values
6. Duplicate Check
7. Data Type Validation
8. Data Quality Checks
9. Export Cleaned Dataset

# 1. Import Libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


# 2. Load Dataset

In [3]:
df = pd.read_csv("dataset/Airline_Delay_Cause.csv")

print(f"Dataset Shape : {df.shape}")

Dataset Shape : (398233, 21)


# 3. Rename Columns

The original dataset contains abbreviated column names. Renaming them improves readability and makes the subsequent analysis easier to understand.

In [4]:
df.rename(columns={
    "arr_flights": "total_arrival_flights",
    "arr_del15": "flights_delayed_over_15min",
    "carrier_ct": "carrier_delay_count",
    "weather_ct": "weather_delay_count",
    "nas_ct": "nas_delay_count",
    "security_ct": "security_delay_count",
    "late_aircraft_ct": "late_aircraft_delay_count",
    "arr_cancelled": "cancelled_flights",
    "arr_diverted": "diverted_flights",
    "arr_delay": "total_arrival_delay_minutes",
    "carrier_delay": "carrier_delay_minutes",
    "weather_delay": "weather_delay_minutes",
    "nas_delay": "nas_delay_minutes",
    "security_delay": "security_delay_minutes",
    "late_aircraft_delay": "late_aircraft_delay_minutes"
}, inplace=True)

print("Columns renamed successfully.")

Columns renamed successfully.


### Observation

The dataset has been standardized by replacing abbreviated column names with descriptive names. This improves code readability and makes subsequent analysis more intuitive without altering the underlying data.

In [5]:
df.head()

,year,month,carrier,carrier_name,airport,airport_name,total_arrival_flights,flights_delayed_over_15min,carrier_delay_count,weather_delay_count,nas_delay_count,security_delay_count,late_aircraft_delay_count,cancelled_flights,diverted_flights,total_arrival_delay_minutes,carrier_delay_minutes,weather_delay_minutes,nas_delay_minutes,security_delay_minutes,late_aircraft_delay_minutes
0,2025,1,G4,Allegiant Air,ELM,"Elmira/Corning, NY: Elmira/Corning Regional",30.0,0.0,0.00,0.0,0.00,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2025,1,G4,Allegiant Air,ELP,"El Paso, TX: El Paso International",2.0,0.0,0.00,0.0,0.00,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2025,1,G4,Allegiant Air,EUG,"Eugene, OR: Mahlon Sweet Field",28.0,8.0,3.74,0.0,1.60,0.0,2.66,2.0,0.0,409.0,236.0,0.0,70.0,0.0,103.0
3,2025,1,G4,Allegiant Air,EVV,"Evansville, IN: Evansville Regional",18.0,1.0,0.00,1.0,0.00,0.0,0.00,0.0,0.0,1075.0,0.0,1075.0,0.0,0.0,0.0
4,2025,1,G4,Allegiant Air,EWR,"Newark, NJ: Newark Liberty International",31.0,5.0,2.17,0.0,2.83,0.0,0.00,1.0,0.0,446.0,336.0,0.0,110.0,0.0,0.0


# 4. Missing Value Analysis

Missing values can affect the quality of analysis and model performance. Before removing or imputing any values, it is important to understand the extent and pattern of missing data.

In [6]:
missing_summary = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage (%)": (df.isnull().sum() / len(df) * 100).round(2)
})

missing_summary = missing_summary[missing_summary["Missing Values"] > 0]

missing_summary

,Missing Values,Percentage (%)
total_arrival_flights,657,0.16
flights_delayed_over_15min,950,0.24
carrier_delay_count,657,0.16
weather_delay_count,657,0.16
nas_delay_count,657,0.16
security_delay_count,657,0.16
late_aircraft_delay_count,657,0.16
cancelled_flights,657,0.16
diverted_flights,657,0.16
total_arrival_delay_minutes,657,0.16


### Observation

The dataset contains missing values in multiple operational metrics. Rather than immediately removing these records, the missing data pattern will be investigated to determine whether the values are random or systematic.

# 5. Investigating Missing Data Patterns

To make an informed cleaning decision, all rows containing missing values are isolated and examined separately.

In [7]:
missing_rows = df[df.isnull().any(axis=1)]

print(f"Rows containing missing values: {len(missing_rows):,}")

Rows containing missing values: 950


In [8]:
missing_rows.head()

,year,month,carrier,carrier_name,airport,airport_name,total_arrival_flights,flights_delayed_over_15min,carrier_delay_count,weather_delay_count,nas_delay_count,security_delay_count,late_aircraft_delay_count,cancelled_flights,diverted_flights,total_arrival_delay_minutes,carrier_delay_minutes,weather_delay_minutes,nas_delay_minutes,security_delay_minutes,late_aircraft_delay_minutes
3208,2024,12,C5,CommuteAir LLC dba CommuteAir,AUS,"Austin, TX: Austin - Bergstrom International",1.0,NaN,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3506,2024,12,G4,Allegiant Air,BTV,"Burlington, VT: Burlington International",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4221,2024,11,C5,CommuteAir LLC dba CommuteAir,DAY,"Dayton, OH: James M Cox/Dayton International",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6899,2024,10,C5,CommuteAir LLC dba CommuteAir,PVD,"Providence, RI: Rhode Island Tf Green Internat...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8371,2024,9,G7,GoJet Airlines LLC d/b/a United Express,HHH,"Hilton Head, SC: Hilton Head Airport",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
missing_rows.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
year,950.0,NaN,NaN,NaN,2016.038947,5.869817,2003.0,2010.0,2020.0,2020.0,2024.0
month,950.0,NaN,NaN,NaN,6.146316,3.177347,1.0,4.0,5.0,9.0,12.0
carrier,950,33,OO,109,NaN,NaN,NaN,NaN,NaN,NaN,NaN
carrier_name,950,41,SkyWest Airlines Inc.,109,NaN,NaN,NaN,NaN,NaN,NaN,NaN
airport,950,254,PVD,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN
airport_name,950,262,"Providence, RI: Theodore Francis Green State",13,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total_arrival_flights,293.0,NaN,NaN,NaN,7.62116,15.370643,1.0,1.0,2.0,7.0,120.0
flights_delayed_over_15min,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
carrier_delay_count,293.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
weather_delay_count,293.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Observation

The preliminary investigation indicates that missing values are concentrated within specific records rather than being randomly distributed across the dataset. Further analysis is required before deciding whether these records should be removed or retained.

# 6. Understanding Missing Data Distribution

Analyze the distribution of missing records across different years and airlines to identify any systematic patterns.

In [10]:
missing_rows["year"].value_counts().sort_index()

year
2003      6
2004     16
2005     18
2006     28
2007     33
2008     82
2009     50
2010     21
2011     18
2012     21
2013     21
2014     23
2015     15
2016     22
2017     15
2018     35
2019     35
2020    338
2021     44
2022     35
2023     52
2024     22
Name: count, dtype: int64

In [11]:
missing_rows["month"].value_counts().sort_index()

month
1      66
2      28
3      61
4     260
5      83
6      66
7      64
8      39
9      69
10    106
11     54
12     54
Name: count, dtype: int64

In [12]:
missing_rows["carrier_name"].value_counts()

carrier_name
SkyWest Airlines Inc.                        109
Comair Inc.                                   90
Allegiant Air                                 64
Mesa Airlines Inc.                            59
ExpressJet Airlines Inc.                      51
Atlantic Southeast Airlines                   46
Endeavor Air Inc.                             44
Delta Air Lines Network                       40
GoJet Airlines LLC d/b/a United Express       39
Air Wisconsin Airlines Corp                   36
Republic Airline                              30
Envoy Air                                     29
Trans States Airlines                         27
Commutair Aka Champlain Enterprises, Inc.     26
Delta Air Lines Inc.                          25
United Air Lines Network                      22
Pinnacle Airlines Inc.                        20
Frontier Airlines Inc.                        20
ExpressJet Airlines LLC                       18
United Air Lines Inc.                         16
Northwe

# 7. Handle Missing Values

The missing value analysis identified two distinct patterns:

- **657 records** have missing values across almost all operational metrics.
- **293 records** contain only the `flights_delayed_over_15min` field as missing.

Based on the data inspection, these records are considered incomplete for meaningful analysis and will be removed to maintain data quality.

Since the missing records account for less than **0.25%** of the dataset, removing them is unlikely to introduce significant bias.

In [13]:
print(f"Dataset Shape Before Cleaning : {df.shape}")

Dataset Shape Before Cleaning : (398233, 21)


In [14]:
df = df.dropna().reset_index(drop=True)

In [15]:
print(f"Dataset Shape After Cleaning : {df.shape}")

Dataset Shape After Cleaning : (397283, 21)


### Observation

A total of **950 incomplete records** were removed from the dataset.

Since these records represented only **0.24%** of the total observations, their removal is not expected to affect the overall analysis while ensuring higher data quality.

# 8. Verify Missing Values

In [16]:
df.isnull().sum()

year                           0
month                          0
carrier                        0
carrier_name                   0
airport                        0
airport_name                   0
total_arrival_flights          0
flights_delayed_over_15min     0
carrier_delay_count            0
weather_delay_count            0
nas_delay_count                0
security_delay_count           0
late_aircraft_delay_count      0
cancelled_flights              0
diverted_flights               0
total_arrival_delay_minutes    0
carrier_delay_minutes          0
weather_delay_minutes          0
nas_delay_minutes              0
security_delay_minutes         0
late_aircraft_delay_minutes    0
dtype: int64

### Observation

The dataset no longer contains missing values and is ready for further validation and preprocessing.

# 9. Check for Duplicate Records

In [17]:
duplicate_rows = df.duplicated().sum()

print(f"Duplicate Records : {duplicate_rows}")

Duplicate Records : 0


# 10. Validate Data Types

Validate the data types of all columns to ensure they are appropriate for analysis and identify any columns that may require conversion.

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397283 entries, 0 to 397282
Data columns (total 21 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   year                         397283 non-null  int64  
 1   month                        397283 non-null  int64  
 2   carrier                      397283 non-null  object 
 3   carrier_name                 397283 non-null  object 
 4   airport                      397283 non-null  object 
 5   airport_name                 397283 non-null  object 
 6   total_arrival_flights        397283 non-null  float64
 7   flights_delayed_over_15min   397283 non-null  float64
 8   carrier_delay_count          397283 non-null  float64
 9   weather_delay_count          397283 non-null  float64
 10  nas_delay_count              397283 non-null  float64
 11  security_delay_count         397283 non-null  float64
 12  late_aircraft_delay_count    397283 non-null  float64
 13 

### Observation

The dataset contains appropriate data types for all features. No additional type conversions are required before analysis.

# 11. Final Data Quality Check

Perform a final validation to ensure the dataset is clean and ready for exploratory data analysis.

In [19]:
print("=" * 50)

print(f"Dataset Shape       : {df.shape}")
print(f"Missing Values      : {df.isnull().sum().sum()}")
print(f"Duplicate Records   : {df.duplicated().sum()}")

print("=" * 50)

Dataset Shape       : (397283, 21)
Missing Values      : 0
Duplicate Records   : 0


### Observation

The dataset has successfully passed all quality checks.

Data cleaning tasks completed:

- Column names standardized
- Missing values removed
- Duplicate records verified
- Data types validated

The dataset is now ready for exploratory data analysis.

# 12. Export Cleaned Dataset

Save the cleaned dataset for use in the subsequent analysis notebooks.

In [21]:
df.to_csv("cleaned_data/airline_delay_cleaned.csv", index=False)

print("Cleaned dataset exported successfully.")

Cleaned dataset exported successfully.
